## 🌍 *Tutorial  #1*: **Learning Air Pollution Dynamics from Data**

Air quality monitoring plays a crucial role in understanding and managing environmental pollution. In many real-world scenarios, we are interested in how a pollutant (such as particulate matter or a gas) spreads over space and time.

In this tutorial, we consider a simplified but insightful setting:

> A pollutant is released into a one-dimensional domain (e.g., along a street or in a simplified atmospheric layer), and spreads over time due to diffusion.

### 🧪 **Physical intuition**

Imagine releasing a small portion of smoke at a specific location. Initially, the concentration is highly localized. Over time:

- the pollutant **spreads out**,
- the peak concentration **decreases**,
- the overall profile becomes **smoother**.

This spreading process is governed by the physics of diffusion.

### 📐 **Problem setting**

We describe the system using:

- **Space**: $x \in [0, 1]$
*| where $x$ can be interpreted as the location along the road*
- **Time**: $t \in [0, 1]$ 
*| where the pollutant release begins at $t=0$*
- **Quantity of interest**: concentration $C(x,t)$
*| where $C\to0$ indicates low concentration and $C\to1$ indicates high concentration*

In this setting, we use **nondimensionalized variables**, meaning that the original physical units of space and time have been scaled out of the problem.

Intuitively, you may think of the spatial coordinate as representing, for example, kilometers along a road, and time as representing hours after the initial pollutant release.

***
### 🤖 **Learning task**

Our goal is to **learn the concentration field** $C(x,t)$ from observed, potentially noisy data.

**What data do we have?**

We have access to measurements of the concentration:

- at a few **selected spatial locations**, where sensors have been installed,
- and only during an **early time period**, since the sensors unfortunately stopped working after a short time.

This means that we do **not** observe the full evolution of the system.

Ultimately, we want to answer the question:
> Can the model still predict how the pollutant spreads at **later times** and across the **the enitre domain**, where no measurements are available?

***
### 📂 Loading and Exploring the Dataset

In this first section, we load the dataset and take a first look at its structure.

The dataset contains samples of the concentration field in long (tidy) format, where each row corresponds to a point in space and time.

In [ ]:
import pandas as pd

data_path = "data/measurements.csv"  

df = pd.read_csv(data_path)

print(f"Number of samples: {len(df)}")
df.head()

Now let us visualize the concentration field using a 2D plot, where the x-axis represents time $t$, the y-axis represents space $x$, and the concentration $C$ is indicated by color.

In [ ]:
from methods.plots import plot_concentration_2d
plot_concentration_2d(df)

From the plot, we can immediately see that data is only available in the time domain $t\le0.25$, and that the measurements may be corrupted by noise. Nevertheless, we will attempt to learn the initial dynamics from the available data using a purely data-driven neural network, and investigate how well it predicts the concentration at later times.

--- 
## 🤖 Implementing a Purely Data-Driven Neural Network (PyTorch)

We now build a simple neural network that learns the mapping:

$
\begin{equation}
(x,t) \mapsto C(x,t)
\end{equation}
$

That is:

- **Input**: spatial coordinate $x$ and time $t$
- **Output**: predicted concentration $C$

This model is **purely data-driven**: it only learns from the observed measurements and has no knowledge of the underlying physics.

## 📦 Data Preparation

First, we prepare the available sensor data for training. We perform a train-test split, allocating $80\%$ of the samples to the training data and $20\%$ to the test data. Afterwards, we convert the resulting pandas DataFrames into **PyTorch tensors**, which are required for training the PyTorch model.

In [ ]:
import torch
from sklearn.model_selection import train_test_split

# Size of test set
test_size = 0.2

# Features and target
X = df[["x", "t"]]
y = df[["C"]]

# Train-test split
X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=42,
    shuffle=True
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_df.values, dtype=torch.float32)
X_test  = torch.tensor(X_test_df.values, dtype=torch.float32)

y_train = torch.tensor(y_train_df.values, dtype=torch.float32)
y_test  = torch.tensor(y_test_df.values, dtype=torch.float32)

# Check shapes
print("Training features:", X_train.shape)
print("Training targets :", y_train.shape)
print("Test features    :", X_test.shape)
print("Test targets     :", y_test.shape)

### 🧠 Define Neural Network Architecture

A small fully connected network is sufficient for this task. Since we use the network for a regression task, hence want to predict concentrations in a **continous manner**, we use a `tanh()` activation for the hidden neurons and a `Sigmoid()` activation for the output layer. 

The `Sigmoid()` activation ensures that all network predictions lie within the range $[0,1]$, which is physically meaningful in this setting, since we are modeling a normalized concentration field.

In [ ]:
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),

            nn.Linear(32, 32),
            nn.Tanh(),

            nn.Linear(32, 32),
            nn.Tanh(),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

### 🔁 Training loop

Let's continue with the implementation of the training loop to learn the initial dynamics from the data.

In [ ]:
def train_model(model, optimizer, criterion,
                X_train, y_train,
                X_test, y_test,
                n_epochs=4000,
                print_every=100):
    """
    Train a PyTorch model and record train/test losses.

    Parameters
    ----------
    model : torch.nn.Module
        Neural network model
    optimizer : torch.optim.Optimizer
        Optimizer (e.g. Adam)
    criterion : loss function
        Loss function (e.g. MSELoss)
    X_train, y_train : torch.Tensor
        Training data
    X_test, y_test : torch.Tensor
        Test data
    n_epochs : int
        Number of training epochs
    print_every : int
        Print progress every n epochs

    Returns
    -------
    history : dict
        Dictionary containing train/test losses
    """

    history = {
        "epoch": [],
        "train_loss": [],
        "test_loss": []
    }

    for epoch in range(1, n_epochs + 1):

        # -------------------------
        # Training step
        # -------------------------
        model.train()
        optimizer.zero_grad()

        y_pred_train = model(X_train)
        train_loss = criterion(y_pred_train, y_train)

        train_loss.backward()
        optimizer.step()

        # -------------------------
        # Evaluation step
        # -------------------------
        model.eval()

        with torch.no_grad():
            y_pred_test = model(X_test)
            test_loss = criterion(y_pred_test, y_test)

        # -------------------------
        # Log history and print current losses
        # -------------------------
        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss.item())
        history["test_loss"].append(test_loss.item())

        # Print progress
        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:5d} | "
                f"Train Loss = {train_loss.item():1.2e} | "
                f"Test Loss = {test_loss.item():1.2e}"
            )

    return history

--- 

## ▶️ Model Training

Now it is time to train the neural network. We first **initialize the model** and define the **training configuration**, including the loss function, optimizer, and number of training epochs.

For this tutorial, we use the standard **mean squared error (MSE)** as the loss function and the **Adam** optimizer, which is a robust and widely used optimization method for neural network training. The initial learning rate we set to the default value of $0.001$.

Once everything is set up, we call the training function to start the learning process and obtain the loss history for both the training and test sets.

In [ ]:
model = NeuralNet()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()
n_epochs = 4000

history = train_model(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    n_epochs=n_epochs,
    print_every=100
)

## 📉 Observing the Learning Curves

It is always good practice to inspect after training the learning curves, which visualize how the training and test losses evolve during the optimization process.

These curves help us assess whether the model is learning effectively, and can reveal important behaviors such as:

- steady improvement during training,
- stagnation or slow convergence,
- **overfitting** (when test loss increases while training loss continues to decrease),
- underfitting (when both losses remain high).

By examining the learning curves, we gain valuable insight into the training dynamics of the neural network.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

plt.plot(history["epoch"], history["train_loss"], label="Train Loss")
plt.plot(history["epoch"], history["test_loss"], label="Test Loss")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.show()

In the figure above, we may indeed observe **indications of overfitting**, since the test loss slightly increases toward the end of training while the training loss continues to decrease. The exact point at which this increase in test loss appears depends on the network initialization, and rerunning the initialization and training code above may lead to slightly different results. This is a consequence of using a **stochastic optimization procedure**.

A likely reason for this behavior is the presence of **noise in the observations**. As training progresses, the network may start fitting not only the underlying concentration dynamics, but also random fluctuations contained in the measurement data. While this can further reduce the training loss, it often harms the model’s ability to generalize to unseen samples, leading to a rise in test loss.

We can address this in several ways:

- leave the model as it is, accepting a small gap between training and test performance,
- rerun the training procedure with a reduced number of epochs (e.g. 1000),
- or apply more advanced regularization techniques that handle this automatically, such as early stopping.

## 🔮 Generating Model Predictions on the Continuous Domain

Now that the neural network has been trained, we would like to inspect how it behaves over the **entire continuous spatio-temporal domain**.

Recall that our problem is defined on:

- Space: $x \in [0,1]$
- Time: $t \in [0,1]$

To visualize the learned concentration field, we create a dense grid of points covering this domain and use the trained model to predict the concentration at each location $(x,t)$.

This allows us to analyze:

- how well the model reconstructs the observed region,
- how it extrapolates to later times,
- whether the predictions remain smooth and physically plausible.

In [ ]:
import numpy as np
import pandas as pd
import torch

# Number of evaluation points
nx = 100
nt = 100

# Continuous domain
x = np.linspace(0.0, 1.0, nx)
t = np.linspace(0.0, 1.0, nt)

# Create full spatio-temporal grid
T_grid, X_grid = np.meshgrid(t, x)

# Flatten grid into coordinate pairs (x, t)
X_eval = np.column_stack([
    X_grid.ravel(),
    T_grid.ravel()
])

# Convert to tensor
X_eval_torch = torch.tensor(X_eval, dtype=torch.float32)

# Predict with trained model
model.eval()

with torch.no_grad():
    C_pred = model(X_eval_torch).cpu().numpy().flatten()

# Store predictions in dataframe
df_pred = pd.DataFrame({
    "x": X_eval[:, 0],
    "t": X_eval[:, 1],
    "C": C_pred
})

plot_concentration_2d(df_pred)

## 🔬 Reality Check: Predictions vs. True Evolution

Well, we were able to use the trained model to generate predictions even beyond the time domain on which it was trained. However, whether these predictions are accurate and, more importantly, physically meaningful still remains to be seen.

Fortunately, our colleague from the lab just called and mentioned that he was able to simulate the concentration evolution on his supercomputer (*grin*). He believes that his results are most likely a close approximation of the true dynamics describing how the concentration evolved after the observed data period.

Let us load this reference data and compare it side by side with our neural network predictions. Additionally, let's compute some performance metric with the reference data. For that, we use the relative $L_2$ error, which is defined according to:

$
\begin{equation}
    \mathrm{rel.}\; L_2 = \frac{\|C_\mathrm{ref} - C_\mathrm{pred}\|_2}{\|C_\mathrm{ref}\|}
\end{equation}
$


In [ ]:
# --------------------------------------------------
# Load reference ("true") dataset
# --------------------------------------------------
df_true = pd.read_csv("data/reference.csv")

# Rename concentration column for clarity
df_true = df_true.rename(columns={"C": "C_true"})

# --------------------------------------------------
# Use coordinates from true dataset for prediction
# --------------------------------------------------
X_true = df_true[["x", "t"]].values
X_true_torch = torch.tensor(X_true, dtype=torch.float32)

# Predict with trained model
model.eval()

with torch.no_grad():
    C_pred = model(X_true_torch).cpu().numpy().flatten()

# Add predictions to dataframe
df_compare = df_true.copy()
df_compare["C_pred"] = C_pred

# Compute absolute residuals
df_compare["residual"] = abs(df_compare["C_true"] - df_compare["C_pred"])

# --------------------------------------------------
# Prepare grids for plotting
# --------------------------------------------------
x_vals = np.sort(df_compare["x"].unique())
t_vals = np.sort(df_compare["t"].unique())

C_true_grid = df_compare.pivot(index="x", columns="t", values="C_true").values
C_pred_grid = df_compare.pivot(index="x", columns="t", values="C_pred").values
R_grid      = df_compare.pivot(index="x", columns="t", values="residual").values

# Common scales
vmin = min(C_true_grid.min(), C_pred_grid.min())
vmax = max(C_true_grid.max(), C_pred_grid.max())

rmax = np.abs(R_grid).max()

# --------------------------------------------------
# Plot side-by-side comparison
# --------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True, constrained_layout=True)

# --- True field
im0 = axes[0].imshow(
    C_true_grid,
    aspect="auto",
    origin="lower",
    extent=[t_vals.min(), t_vals.max(), x_vals.min(), x_vals.max()],
    vmin=vmin,
    vmax=vmax
)
axes[0].set_title("Reference Data")
axes[0].set_xlabel("Time t")
axes[0].set_ylabel("Space x")

# --- Prediction
im1 = axes[1].imshow(
    C_pred_grid,
    aspect="auto",
    origin="lower",
    extent=[t_vals.min(), t_vals.max(), x_vals.min(), x_vals.max()],
    vmin=vmin,
    vmax=vmax
)
axes[1].set_title("Neural Network Prediction")
axes[1].set_xlabel("Time t")

# --- Residuals
im2 = axes[2].imshow(
    R_grid,
    aspect="auto",
    origin="lower",
    extent=[t_vals.min(), t_vals.max(), x_vals.min(), x_vals.max()],
    vmin=0,
    vmax=rmax
)
axes[2].set_title("Absolute Residuals (True - Predicted)")
axes[2].set_xlabel("Time t")

# Colorbars
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="Concentration")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="Concentration")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label="Residual")

plt.show()

# --------------------------------------------------
# rel. L2 error computation
# --------------------------------------------------
# Flatten arrays / columns
C_true = df_compare["C_true"].values
C_pred = df_compare["C_pred"].values

relative_l2_error = np.linalg.norm(C_true - C_pred) / np.linalg.norm(C_true)

print(f"Relative L2 Error: {100 * relative_l2_error:.1f}%")

The figures above, together with the relative $L_2$ error, indicate that although we were able to learn some aspects of the underlying dynamics from the available data, the mismatch between the model predictions and the true solution is still considerable. A relative $L_2$ error greater than 15% is often an indication of a noticeable discrepancy.

There are several design choices we could explore to improve the performance of the purely data-driven model, such as:

- trying different hidden-layer activation functions,
- increasing or decreasing the network size,
- adjusting the learning rate,
- or tuning other training hyperparameters.

However, we would soon realize that this task remains challenging to solve using a purely data-driven approach. The reason is simple: the model only has access to a limited amount of noisy data, and no information about the physical mechanism governing the system. (Using the reference data for training would, of course, be cheating—we did not know it in advance.)

But wait... we do know from physics that the concentration dynamics should follow a simple **diffusion equation**.

Perhaps we can use this knowledge to **physics-inform** our neural network.

*Let's continue with the next tutorial...*

---